# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You'll use Croissant schema 
URLs and refer to every entity by its `@id` throughout the notebook.

### Dataset Source
The Croissant schema for the FAIR² dataset: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

> All references to record sets, fields, and columns below are **by their `@id`** as required by the Croissant specification and the mlcroissant API.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Dataset metadata
ds = mlc.Dataset(croissant_url)

metadata = ds.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'n/a')}, Version: {getattr(metadata, 'version', 'n/a')}")
print(f"Authors: {getattr(metadata, 'author', 'n/a')}")

## 2. Data Overview
Review available record sets and their `@id` values. 
Each record set describes a collection of records (e.g. tables, main outputs, or entities), each field or column also has a unique `@id`.

In [ ]:
# Display all record sets by @id
print("Available record sets and their field structure:\n")

record_sets = list(ds.metadata.record_sets)
for rs in record_sets:
    print(f"RecordSet: @id={rs.id}, name={getattr(rs, 'name', '')}")
    if getattr(rs, 'fields', None):
        for fld in rs.fields:
            name = getattr(fld, 'name', fld.id)
            dtype = getattr(fld, 'data_type', 'unknown')
            print(f"  Field: @id={fld.id}, name={name}, data_type={dtype}")
    print()

## 3. Data Extraction

We'll load all record sets by their `@id` into pandas dataframes for easy manipulation. Please select a `@id` shown above for further analysis if you wish to work with a specific record set.

> **Note:** All data is referred to by `@id` as per the Croissant/FAIR² guidelines.

In [ ]:
# Extract all record sets into a dict of DataFrames for exploration
dataframes = dict()
record_set_ids = [rs.id for rs in ds.metadata.record_sets]

for record_set_id in record_set_ids:
    # We load all records for each record set
    df = pd.DataFrame(list(ds.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"Loaded record set @id={record_set_id}: shape={df.shape}")

# Show columns for each record set (by @id)
for rsid, df in dataframes.items():
    print(f"\n@id={rsid} columns: {list(df.columns)}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)

We'll perform some sample operations:
- Filtering numeric fields (referenced by their `@id`)
- Normalizing numeric columns
- Grouping by a categorical field

> Please **adjust the `record_set_id`, `numeric_field_id`, and `group_field_id` to match real available @id values discovered above.**

Below is a template using the first non-empty record set and inferring numeric/categorical fields automatically.

In [ ]:
# Choose one record set with data
chosen_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        chosen_record_set_id = rsid
        break

if chosen_record_set_id is None:
    print('No non-empty record set found for EDA.')
else:
    df = dataframes[chosen_record_set_id].copy()
    # Try to automatically select a numeric field by type or column name
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Attempt heuristic for numeric columns (columns with 'value', 'score', etc)
        numeric_candidates = [col for col in df.columns if any(s in col.lower() for s in ['score', 'value', 'likelihood', 'coef', 'err'])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f'Using numeric field @id: {numeric_field_id}')
        # Choose a threshold for demonstration
        threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Choose a group field (try a non-numeric)
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"\nGrouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
    else:
        print('No numeric field found in selected record set.')

## 5. Visualization

Plot a histogram of a numeric field (referenced by `@id`), and a bar chart of the mean grouped values if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id is not None and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion

- Demonstrated dataset loading directly from the FAIR² Croissant schema URL via `mlcroissant`.
- Enumerated record sets, fields, all referenced strictly by `@id`.
- Extracted data from record sets into pandas DataFrames for analysis.
- Conducted example EDA: filtering, normalization, grouping, and visualization.

👉 You can further branch out your exploration by referencing other record sets or fields by their `@id` as shown in cell outputs above.

[mlcroissant documentation](https://mlcroissant.readthedocs.io/)